<a href="https://colab.research.google.com/github/simionRiccard0/Deep-Learning-Project/blob/main/viral_genomes_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DNA sequences for identifying viral
genomes in human samples.

In [1]:
!git clone https://github.com/NeuroCSUT/ViraMiner.git

Cloning into 'ViraMiner'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 78 (delta 10), reused 10 (delta 10), pack-reused 63 (from 1)
Receiving objects: 100% (78/78), 247.04 MiB | 22.15 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Updating files: 100% (57/57), done.


In [2]:
import os
os.listdir('ViraMiner/data/DNA_data/')

['exp11_2014_G7.csv',
 'create_LOO_set.py',
 'exp4_2014_B.csv',
 'fullset_test.csv',
 'exp13_2014_N1.csv',
 'exp15_2014_P.csv',
 'exp0_2011_G5.csv',
 'fullset_train.csv',
 'exp3_2013_H4.csv',
 'create_dataset.py',
 'exp8_2014_G1.csv',
 'exp12_2014_J1.csv',
 'exp18_2015_F.csv',
 'exp10_2014_G6.csv',
 'exp6_2014_E1.csv',
 'exp7_2014_F1.csv',
 'exp1_2011_N19.csv',
 'fullset_validation.csv',
 'exp16_2014_R1.csv',
 'exp9_2014_G5.csv',
 'exp14_2014_O.csv',
 'exp17_2015_F2.csv',
 'exp2_2013_E2_SCC.csv',
 'exp5_2014_D3.csv']

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [4]:
import pandas as pd

train = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_train.csv',
    header=None
)

val = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_validation.csv',
    header=None
)

test = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_test.csv',
    header=None
)

print("Train shape:", train.shape)
print(train.head())

Train shape: (211239, 3)
                                                 0  \
0  seq1006848_2014_P_Lagheden_Matti-HPV-vacc-CIN31   
1                     seq4116690_2014_G6_Hultin_MS   
2            seq360982_2014_F1_PARAFFINBLANKBLOCKS   
3             seq1135576_2011_N19_VIRASKINFAPMISEQ   
4                        seq277_2014_G6_Hultin_MS8   

                                                   1  2  
0  CAAGCCAAGATTTTCTCGCGTCACACTACTCATGACCATTGTATTA...  0  
1  AACGAAGCACGGGCCGAGAGATTGAGGAACCAAGGTCCAGCTCTAG...  0  
2  TAGTGGGTGAGGTTTCTATTTCCATAATGATCTCGCCTCAATTACT...  0  
3  ATATGACCATTCTTGCAAGGTAACACAGGTACATTTTCACAAAGTG...  0  
4  GGTCTTAAAACAACAGAAATTTTTTCCATCACAGTTGCAGAAATTA...  0  


In [5]:
print("Sequence length:", len(train[1][0]))

print("\nClass balance:")
print(train[2].value_counts())

Sequence length: 300

Class balance:
2
0    206773
1      4466
Name: count, dtype: int64


DNA sequences encoding:

In [6]:
import torch
from torch.utils.data import Dataset
import random # Added for sequence augmentation

#DNA one-hot encoding
mapping = {
    "A": [1,0,0,0],
    "C": [0,1,0,0],
    "G": [0,0,1,0],
    "T": [0,0,0,1],
    "N": [0,0,0,0]
}

MAX_LEN = 300

def encode_sequence(seq, augment=False):
    complement = {'A':'T','T':'A','C':'G','G':'C','N':'N'}
    if augment and random.random() > 0.5:
        seq = ''.join(complement.get(base, 'N') for base in reversed(seq))
    if augment:
        shift = random.randint(-5, 5)
        seq = seq[max(0, shift):] if shift > 0 else 'N'*(-shift) + seq

    seq = seq[:MAX_LEN]

    encoded = [
        mapping.get(base, [0,0,0,0])
        for base in seq
    ]

    while len(encoded) < MAX_LEN:
        encoded.append([0,0,0,0])

    return encoded


class DNADataset(Dataset):

    def __init__(self, dataframe, augment=False):

        self.sequences = dataframe[1].values
        self.labels = dataframe[2].values
        self.augment = augment # Added for augmentation

    def __len__(self):

        return len(self.sequences)

    def __getitem__(self, idx):

        seq = self.sequences[idx]
        label = self.labels[idx]

        x = torch.tensor(
            encode_sequence(seq, self.augment), # Pass augment flag
            dtype=torch.float32
        )

        y = torch.tensor(
            label,
            dtype=torch.float32
        )

        return x, y

Preparing the dataset (DataLoader):

In [7]:
from torch.utils.data import DataLoader

train_dataset = DNADataset(train, augment=True) # Added augment=True
val_dataset   = DNADataset(val)
test_dataset  = DNADataset(test)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 300, 4])
torch.Size([128])


CNN Branches

In [8]:
import torch.nn as nn
import torch.nn.functional as F

class PatternBranch(nn.Module):
    def __init__(self, dropout): # Added dropout parameter
        super().__init__()
        self.conv6  = nn.Conv1d(4, 512, 6,  padding='same')
        self.conv12 = nn.Conv1d(4, 512, 12, padding='same')
        self.conv18 = nn.Conv1d(4, 512, 18, padding='same')
        self.conv24 = nn.Conv1d(4, 512, 24, padding='same')

        total = 512 * 4
        self.bn   = nn.BatchNorm1d(total)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(total, 512),
            nn.GELU(),
            nn.Dropout(dropout) # Used passed dropout
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        branches = [
            F.gelu(self.conv6(x)),
            F.gelu(self.conv12(x)),
            F.gelu(self.conv18(x)),
            F.gelu(self.conv24(x)),
        ]
        x = torch.cat(branches, dim=1)
        x = self.bn(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

In [9]:
class FrequencyBranch(nn.Module):
    def __init__(self, dropout): # Added dropout parameter
        super().__init__()
        self.conv6  = nn.Conv1d(4, 256, 6,  padding='same')
        self.conv12 = nn.Conv1d(4, 256, 12, padding='same')
        self.conv18 = nn.Conv1d(4, 256, 18, padding='same')
        self.conv24 = nn.Conv1d(4, 256, 24, padding='same')

        total = 256 * 4
        self.bn   = nn.BatchNorm1d(total)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(total, 512),
            nn.GELU(),
            nn.Dropout(dropout) # Used passed dropout
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        branches = [
            F.gelu(self.conv6(x)),
            F.gelu(self.conv12(x)),
            F.gelu(self.conv18(x)),
            F.gelu(self.conv24(x)),
        ]
        x = torch.cat(branches, dim=1)
        x = self.bn(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

Transformer branch

In [10]:
class TransformerBranch(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=4, max_len=300, dropout=0.1): # Added dropout parameter
        super().__init__()
        self.embedding = nn.Linear(4, d_model)
        self.pos_enc   = nn.Embedding(max_len + 1, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, # Used passed dropout
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.out  = nn.Linear(d_model, 512)

    def forward(self, x):
        B, L, _ = x.shape
        x = self.embedding(x)
        positions = torch.arange(1, L + 1, device=x.device).unsqueeze(0)
        x = x + self.pos_enc(positions)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x = self.transformer(x)
        x = self.norm(x)
        return self.out(x[:, 0, :])

ViraExplorer Model (CNN Pattern branch + CNN Frequency branch + Transformer)

In [11]:
class ViraExplorer(nn.Module):
    def __init__(self, cnn_dropout=0.25, transformer_dropout=0.1): # Modified dropout parameters
        super().__init__()
        self.pattern     = PatternBranch(cnn_dropout)     # Used cnn_dropout
        self.frequency   = FrequencyBranch(cnn_dropout)   # Used cnn_dropout
        self.transformer = TransformerBranch(dropout=transformer_dropout) # Used transformer_dropout
        self.fc = nn.Sequential(
            nn.Linear(1536, 512),
            nn.GELU(),
            nn.Dropout(cnn_dropout), # Used cnn_dropout
            nn.LayerNorm(512),
            nn.Linear(512, 128),
            nn.GELU(),
            nn.Dropout(cnn_dropout), # Used cnn_dropout
            nn.Linear(128, 1)
        )

    def forward(self, x):
        p = self.pattern(x)
        f = self.frequency(x)
        t = self.transformer(x)
        x = torch.cat([p, f, t], dim=1)
        return self.fc(x)

Focal Loss

In [12]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05): # Added smoothing parameter
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing # Stored smoothing parameter

    def forward(self, logits, targets):
        targets = targets.unsqueeze(1)
        # Apply label smoothing
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing # Applied label smoothing
        bce     = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none') # Used smoothed targets
        pt           = torch.exp(-bce)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

Training Loop

In [ ]:
import os
import math
import torch
import numpy as np
from sklearn.metrics import roc_auc_score

SAVE_DIR = "/content/"

BEST_MODEL_PATH  = os.path.join(SAVE_DIR, "best_viraexplorer_v2.pth")
CHECKPOINT_PATH  = os.path.join(SAVE_DIR, "checkpoint.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model     = ViraExplorer(cnn_dropout=0.25, transformer_dropout=0.1).to(device)
criterion = FocalLoss(alpha=0.75, gamma=2.0, smoothing=0.05)

EPOCHS       = 120
WARMUP_EPOCHS = 5

transformer_params = list(model.transformer.parameters())
other_params = (list(model.pattern.parameters()) +
                list(model.frequency.parameters()) +
                list(model.fc.parameters()))

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return epoch / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))

optimizer = torch.optim.AdamW([
    {"params": other_params,       "lr": 1e-4},
    {"params": transformer_params, "lr": 5e-5},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler('cuda')

best_auc    = 0
patience    = 15
counter     = 0
START_EPOCH = 0

for p in model.transformer.parameters():
    p.requires_grad = False

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    best_auc    = ckpt['best_auc']
    counter     = ckpt['counter']
    START_EPOCH = ckpt['epoch'] + 1
    if START_EPOCH >= WARMUP_EPOCHS:
        for p in model.transformer.parameters():
            p.requires_grad = True
        print("Transformer already unfrozen (past warmup)")
    print(f"Resumed from epoch {START_EPOCH} | Best AUC so far: {best_auc:.4f}")
else:
    print("No checkpoint found — starting from scratch")

print(f"Starting from epoch {START_EPOCH}")
for epoch in range(START_EPOCH, EPOCHS):


    if epoch == WARMUP_EPOCHS:
        for p in model.transformer.parameters():
            p.requires_grad = True
        print("Transformer unfrozen")

    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            outputs = model(X_batch)
            loss    = criterion(outputs, y_batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    scheduler.step()


    model.eval()
    preds, targets_list = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            probs = torch.sigmoid(model(X_batch.to(device)))
            preds.extend(probs.cpu().numpy().flatten())
            targets_list.extend(y_batch.numpy())

    auc = roc_auc_score(targets_list, preds)
    print(f"Epoch {epoch+1:3d} | Loss: {total_loss/len(train_loader):.4f} | Val AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        counter  = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_auc':        best_auc,
            'counter':         counter,
        }, CHECKPOINT_PATH)
        print(f"  ✓ Saved new best: {best_auc:.4f}")
    else:
        counter += 1
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'best_auc':        best_auc,
            'counter':         counter,
        }, CHECKPOINT_PATH)
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1} | Best Val AUC: {best_auc:.4f}")
            break

print(f"\nTraining complete | Best Val AUC: {best_auc:.4f}")


from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt

print("\nRunning test set evaluation...")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

preds, targets = [], []
with torch.no_grad():
    for X, y in test_loader:
        probs = torch.sigmoid(model(X.to(device)))
        preds.extend(probs.cpu().numpy().flatten())
        targets.extend(y.numpy())

preds   = np.array(preds)
targets = np.array(targets)

auroc = roc_auc_score(targets, preds)
auprc = average_precision_score(targets, preds)

prec, rec, thresh = precision_recall_curve(targets, preds)
f1s         = 2*prec*rec / (prec+rec+1e-8)
best_thresh = thresh[np.argmax(f1s)]
preds_bin   = (preds >= best_thresh).astype(int)

tn, fp, fn, tp = confusion_matrix(targets, preds_bin).ravel()

print("=" * 45)
print(f"  Test AUROC        : {auroc:.4f}   (paper: 0.923)")
print(f"  Test AUPRC        : {auprc:.4f}")
print(f"  Best threshold    : {best_thresh:.3f}")
print(f"  Precision (virus) : {tp/(tp+fp):.4f}")
print(f"  Recall    (virus) : {tp/(tp+fn):.4f}")
print(f"  F1        (virus) : {2*tp/(2*tp+fp+fn):.4f}")
print("=" * 45)
print(classification_report(targets, preds_bin, target_names=["non-virus","virus"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr, tpr, _ = roc_curve(targets, preds)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'ViraExplorer (AUC={auroc:.3f})')
axes[0].plot([0,1],[0,1],'k--', label='Random (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Test Set'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'ViraExplorer (AP={auprc:.3f})')
axes[1].axhline(y=0.9, color='red', linestyle='--', alpha=0.7, label='90% precision')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — Test Set'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, "test_evaluation.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"\nPlot saved to {plot_path}")

Device: cuda
No checkpoint found — starting from scratch
Starting from epoch 0


/tmp/ipykernel_647/2487011402.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch   1 | Loss: 0.1184 | Val AUC: 0.5231
  ✓ Saved new best: 0.5231
Epoch   2 | Loss: 0.0261 | Val AUC: 0.6992
  ✓ Saved new best: 0.6992
Epoch   3 | Loss: 0.0254 | Val AUC: 0.7342
  ✓ Saved new best: 0.7342
Epoch   4 | Loss: 0.0230 | Val AUC: 0.8183
  ✓ Saved new best: 0.8183
Epoch   5 | Loss: 0.0210 | Val AUC: 0.8251
  ✓ Saved new best: 0.8251
Transformer unfrozen
Epoch   6 | Loss: 0.0207 | Val AUC: 0.8367
  ✓ Saved new best: 0.8367
Epoch   7 | Loss: 0.0204 | Val AUC: 0.8358
Epoch   8 | Loss: 0.0200 | Val AUC: 0.8528
  ✓ Saved new best: 0.8528
Epoch   9 | Loss: 0.0194 | Val AUC: 0.8589
  ✓ Saved new best: 0.8589
Epoch  10 | Loss: 0.0189 | Val AUC: 0.8598
  ✓ Saved new best: 0.8598
Epoch  11 | Loss: 0.0187 | Val AUC: 0.8675
  ✓ Saved new best: 0.8675
Epoch  12 | Loss: 0.0184 | Val AUC: 0.8722
  ✓ Saved new best: 0.8722
Epoch  13 | Loss: 0.0182 | Val AUC: 0.8719
Epoch  14 | Loss: 0.0180 | Val AUC: 0.8788
  ✓ Saved new best: 0.8788
Epoch  15 | Loss: 0.0178 | Val AUC: 0.8812
  ✓ Saved 